# Check an Analyst report

Use this after generating a candidate. The evidence notebook checks the inputs;
this notebook checks what the Analyst said about them.

Set `attempt_id` below, restart the kernel, and run all cells. Nothing here calls
a model, changes a report, or publishes files. There are no generated reports yet.

In [ ]:
import json
import sys
from pathlib import Path
from pprint import pprint

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/benchmark/private/packets").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from inside the agent_test repository.")
folder = repo / "data/benchmark"
sys.path.insert(0, str(repo))
from src.benchmark.prepare import check_report, assert_masked
from src.benchmark.schemas import Packet, Report

attempt_id = ""  # Paste the folder name printed by --generate.
attempt = packet = report = None
if not attempt_id:
    print("Choose an attempt_id after generating a report. Available candidates:")
    for path in sorted((folder / "private/attempts").glob("*/candidate.json")):
        print(path.parent.name)
else:
    attempt_folder = folder / "private/attempts" / attempt_id
    attempt = json.loads((attempt_folder / "attempt.json").read_text(encoding="utf-8"))
    report = json.loads((attempt_folder / "candidate.json").read_text(encoding="utf-8"))
    # Use the evidence actually sent for this attempt, even if today's packet changed.
    sent = json.loads(attempt["request"]["messages"][1]["content"])
    packet = sent["evidence_packet"]
    print("Candidate:", attempt_folder / "candidate.json")
    print("Packet:", packet["packet_id"], "| Report:", report["report_id"])
    print("Requested model:", attempt["request"]["model"])
    print("Returned model:", attempt.get("returned_model"))

## Read the report

Check the recommendation, claims, reasoning, and confidence wording.

In [ ]:
if report is not None:
    pprint(report, sort_dicts=False)

## Claims and their evidence

Does the cited evidence support each claim? Check for missing counter-evidence too.

In [ ]:
if report is not None:
    evidence_by_id = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
    for claim in report["claims"]:
        print("\nCLAIM", claim["claim_id"], claim["text"])
        print("Numerical assertion:", claim.get("check_spec"))
        for evidence_id in claim["evidence_ids"]:
            print("\nEVIDENCE", evidence_id)
            print(evidence_by_id.get(evidence_id, "MISSING EVIDENCE ID"))
    print("\nAll evidence IDs:", list(evidence_by_id))
    print("Read the full packet too, including evidence that was not cited:")
    pprint(packet, sort_dicts=False)

## Automatic checks

These check structure, citations, structured arithmetic, and known identity leaks.
They do not prove that the reasoning is sound. Check the listed review points yourself.

In [ ]:
if report is not None:
    aliases = json.loads((folder / "private/aliases.json").read_text(encoding="utf-8"))
    try:
        assert_masked(report, aliases)
        checks = check_report(Packet.model_validate(packet), Report.model_validate(report))
        pprint(checks, sort_dicts=False)
    except ValueError as error:
        print("CHECK FAILED:", error)

## Review notes

Edit this cell. Check facts, arithmetic, citations, reasoning, counter-evidence,
masking, and neutral confidence wording. Use the evidence notebook and PDF when needed.

- Attempt ID: pending
- Packet ID: pending
- Report ID: pending
- reviewer_1: pending; notes:
- reviewer_2: pending; notes:

Each reviewer writes `approved` or `changes_requested` with any notes. If changes
are needed, regenerate with masked feedback; do not edit the candidate by hand.
Save a copy of this notebook as `private/reviews/<report_id>.ipynb` so another
report's review does not replace these notes. The saved notebook is the review record;
you do not need to fill in the generated `review.json` checklist as well.

## When both reviewers are satisfied

After resolving automatic errors and review points, manually copy the unchanged
`candidate.json` to `public/reports/<report_id>.json`. Keep the candidate and this
review notebook privately. Confirm both initial reports used the same chosen model.

For the paired benchmark, make a separate flawed copy after approving the clean
report. Review the intended change before saving the dataset version. Pair creation
is not implemented yet. A dataset version is just a saved copy under `releases/v1/`;
keep it unchanged once experiments start.